# 🛡️ VoiceShield: Comprehensive Golden Vault Evaluation Benchmark
### Complete 1,818-Sample Out-of-Sample Performance Audit on NVIDIA GPU (Google Colab)

This notebook automates testing on **all 1,818 audios in the official VoiceShield Golden Vault**:
- 🎙️ **818 Real Human Voices** (409 English + 409 Hindi test audio clips).
- 🤖 **1,000 AI Synthetic Voices** (500 English + 500 Hindi across multi-speaker vocoders).
- 📡 **Acoustic Diversity**: Clean studio speech, telephone narrowband filters (300Hz-3400Hz), room reverberation / replay echo, and ambient background noise.
- ⚡ **Native Direct Inference**: Fully utilizes Wav2Vec 2.0 XLSR-53 continuous sequence feature extraction without artificial zero-padding slicing.
- 📊 **Full Metrics Export**: Computes Accuracy, Precision, Recall, Specificity, F1-Score, False Positive Rate (FPR), False Negative Rate (FNR), per-language breakdowns, and exports a CSV + JSON audit report for academic examiners.

In [ ]:
# =====================================================
# Step 0: Hardware Verification (NVIDIA GPU Check)
# =====================================================
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Running on CPU. For maximum speed, go to Runtime > Change runtime type > T4 GPU.")

In [ ]:
# =====================================================
# Step 1: Install Required Dependencies
# =====================================================
!pip install -q transformers soundfile librosa huggingface_hub pandas tqdm scikit-learn tabulate
print("✅ All dependencies installed successfully!")

In [ ]:
# =====================================================
# Step 2: Download All 1,818 Golden Vault Audios
# Dataset: ArmanSandhu1729/voiceshield-golden-vault
# =====================================================
import os
import glob
from huggingface_hub import snapshot_download

REPO_ID = "ArmanSandhu1729/voiceshield-golden-vault"
print(f"📥 Downloading all Golden Vault audios from {REPO_ID}...")

dataset_dir = snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    allow_patterns=["test-Bi-new-2k-01/*"],
    local_dir="golden_vault_local"
)

real_files = sorted(glob.glob("golden_vault_local/test-Bi-new-2k-01/real/*.wav"))
fake_files = sorted(glob.glob("golden_vault_local/test-Bi-new-2k-01/fake/*.wav"))

print(f"\n✅ Download complete!")
print(f"   - Real Human Audios: {len(real_files)}")
print(f"   - AI Synthetic Audios: {len(fake_files)}")
print(f"   - Total Files Ready for Inference: {len(real_files) + len(fake_files)}")

In [ ]:
# =====================================================
# Step 3: Load VoiceShield Hardened Deepfake Model onto GPU
# Model: ArmanSandhu1729/voiceshield-hindi-english-deepfake-v3-hardened
# =====================================================
import torch
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

MODEL_ID = "ArmanSandhu1729/voiceshield-hindi-english-deepfake-v3-hardened"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"🧠 Loading {MODEL_ID} onto {device}...")
extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)
model = AutoModelForAudioClassification.from_pretrained(MODEL_ID).to(device)
model.eval()

# Resolve fake label index
id2label = getattr(model.config, "id2label", {0: "fake", 1: "real"})
fake_idx = 0
for idx, lbl in id2label.items():
    if "fake" in str(lbl).lower() or "ai" in str(lbl).lower() or str(idx) == "1":
        fake_idx = int(idx)
        break
real_idx = 1 - fake_idx
print(f"✅ Model loaded successfully! Class Mapping: fake_idx={fake_idx}, real_idx={real_idx} ({id2label})")

In [ ]:
# =====================================================
# Step 4: High-Speed GPU Batch Inference Across All Files
# (Direct sequence feature extraction without zero-padding)
# =====================================================
import time
import random
import numpy as np
import soundfile as sf
import librosa
from tqdm.auto import tqdm

eval_items = [(f, "real") for f in real_files] + [(f, "fake") for f in fake_files]
random.seed(42)
random.shuffle(eval_items)

print(f"⚡ Starting GPU Evaluation across {len(eval_items)} Golden Vault audios...")

results = []
tp, tn, fp, fn = 0, 0, 0, 0
hi_total, hi_correct = 0, 0
en_total, en_correct = 0, 0
latencies = []

t_eval_start = time.time()

for idx, (fpath, ground_truth) in enumerate(tqdm(eval_items, desc="Auditing Golden Vault"), 1):
    fname = os.path.basename(fpath)
    lang = "hindi" if "hi_" in fname else "english"
    
    speech, sr = sf.read(fpath)
    if len(speech.shape) > 1:
        speech = np.mean(speech, axis=-1)
    if sr != 16000:
        speech = librosa.resample(speech, orig_sr=sr, target_sr=16000)
        sr = 16000
        
    dur = len(speech) / sr
    
    t0 = time.time()
    inputs = extractor(speech, sampling_rate=16000, return_tensors="pt").to(device)
    
    with torch.no_grad():
        logits = model(inputs.input_values).logits.squeeze()
        probs = torch.softmax(logits, dim=-1).cpu().tolist()
        if isinstance(probs, float):
            probs = [probs, 1.0 - probs]
            
    lat = (time.time() - t0) * 1000.0
    latencies.append(lat)
    
    fake_prob = float(probs[fake_idx])
    real_prob = float(probs[real_idx])
    pred = "AI_GENERATED" if fake_prob >= 0.50 else "HUMAN_SPEECH"
    
    is_correct = False
    if ground_truth == "fake":
        if pred == "AI_GENERATED":
            tp += 1
            is_correct = True
        else:
            fn += 1
    else:
        if pred == "HUMAN_SPEECH":
            tn += 1
            is_correct = True
        else:
            fp += 1
            
    if lang == "hindi":
        hi_total += 1
        if is_correct: hi_correct += 1
    else:
        en_total += 1
        if is_correct: en_correct += 1
        
    results.append({
        "filename": fname,
        "language": lang,
        "ground_truth": ground_truth,
        "prediction": pred,
        "is_correct": is_correct,
        "real_score": round(real_prob, 4),
        "fake_score": round(fake_prob, 4),
        "duration_sec": round(dur, 2),
        "latency_ms": round(lat, 1)
    })

total_elapsed = time.time() - t_eval_start
print(f"\n🎉 Full inference run completed in {total_elapsed:.1f} seconds! (Average: {np.mean(latencies):.1f} ms/audio)")

In [ ]:
# =====================================================
# Step 5: Metrics Synthesis & Statistical Evaluation
# =====================================================
import pandas as pd
from tabulate import tabulate

total_evaluated = len(results)
accuracy = ((tp + tn) / total_evaluated) * 100.0
precision = (tp / (tp + fp)) * 100.0 if (tp + fp) > 0 else 0.0
recall = (tp / (tp + fn)) * 100.0 if (tp + fn) > 0 else 0.0
specificity = (tn / (tn + fp)) * 100.0 if (tn + fp) > 0 else 0.0
f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
fpr = (fp / (fp + tn)) * 100.0 if (fp + tn) > 0 else 0.0
fnr = (fn / (fn + tp)) * 100.0 if (fn + tp) > 0 else 0.0

hi_acc = (hi_correct / hi_total * 100.0) if hi_total > 0 else 0.0
en_acc = (en_correct / en_total * 100.0) if en_total > 0 else 0.0

metrics_table = [
    ["Overall Accuracy", f"{accuracy:.2f}%"],
    ["Precision (AI Detection)", f"{precision:.2f}%"],
    ["Recall / Sensitivity", f"{recall:.2f}%"],
    ["Specificity (Human Preservation)", f"{specificity:.2f}%"],
    ["F1-Score", f"{f1:.2f}%"],
    ["False Positive Rate (FPR)", f"{fpr:.2f}%"],
    ["False Negative Rate (FNR)", f"{fnr:.2f}%"],
    ["Hindi Accuracy", f"{hi_acc:.2f}% ({hi_correct}/{hi_total})"],
    ["English Accuracy", f"{en_acc:.2f}% ({en_correct}/{en_total})"],
    ["Average Latency (GPU)", f"{np.mean(latencies):.1f} ms"]
]

print("=" * 60)
print("🛡️ VOICESHIELD GOLDEN VAULT EVALUATION REPORT")
print("=" * 60)
print(tabulate(metrics_table, headers=["Metric", "Performance Score"], tablefmt="fancy_grid"))

confusion_table = [
    ["Actual AI Synthetic", f"TP = {tp} (Correctly Flagged)", f"FN = {fn} (Missed Spoofs)"],
    ["Actual Real Human", f"FP = {fp} (False Alarms)", f"TN = {tn} (Verified Human)"]
]
print("\n📊 CONFUSION MATRIX:")
print(tabulate(confusion_table, headers=["Ground Truth", "Predicted AI", "Predicted Human"], tablefmt="grid"))

In [ ]:
# =====================================================
# Step 6: Export Results to CSV and JSON for Teachers
# =====================================================
import json
from google.colab import files

df_results = pd.DataFrame(results)
csv_filename = "voiceshield_golden_vault_predictions.csv"
df_results.to_csv(csv_filename, index=False)

summary_data = {
    "model_id": MODEL_ID,
    "dataset": REPO_ID,
    "total_samples": total_evaluated,
    "total_time_seconds": round(total_elapsed, 2),
    "average_latency_ms": round(float(np.mean(latencies)), 2),
    "overall_accuracy": round(accuracy, 2),
    "precision": round(precision, 2),
    "recall": round(recall, 2),
    "specificity": round(specificity, 2),
    "f1_score": round(f1, 2),
    "false_positive_rate": round(fpr, 2),
    "false_negative_rate": round(fnr, 2),
    "hindi_accuracy": round(hi_acc, 2),
    "english_accuracy": round(en_acc, 2),
    "confusion_matrix": {"tp": tp, "tn": tn, "fp": fp, "fn": fn}
}

json_filename = "voiceshield_golden_vault_summary.json"
with open(json_filename, "w") as f:
    json.dump(summary_data, f, indent=2)

print(f"✅ Reports saved to '{csv_filename}' and '{json_filename}'.")
print("📥 Triggering automatic download of evaluation reports...")
files.download(csv_filename)
files.download(json_filename)